In [ ]:
"""
10_player_impact_analysis.py -- Scout AI Interactive Waterfall Analysis

Looks up a single player and explains their specific market valuation
using a SHAP waterfall plot. Routes the player to the appropriate model
based on their transfer history.
"""

import os
import sys
import pandas as pd
import numpy as np
import joblib
import shap
import matplotlib
matplotlib.use('Agg')  # Save to file without opening interactive window
import matplotlib.pyplot as plt
from sqlalchemy import create_engine
from dotenv import load_dotenv, find_dotenv

# ==========================================
# 0. CONFIG & SETUP
# ==========================================

# Load environment variables
load_dotenv(find_dotenv())

MODELS_DIR = "models"
IMAGES_DIR = "images"

# Ensure output directory exists
os.makedirs(IMAGES_DIR, exist_ok=True)

# ==========================================
# 1. CONSTANTS & FEATURE LISTS
# ==========================================

FULL_FEATURES = [
    'age', 'height_in_cm', 'total_appearances', 'minutes_per_match',
    'total_goals', 'total_assists', 'goals_per_90', 'assists_per_90',
    'total_yellow_cards', 'total_red_cards', 'international_caps', 'international_goals',
    'club_squad_size', 'club_avg_age', 'stadium_seats', 'foreigners_percentage',
    'contract_months_remaining', 'wonderkid_hype', 'league_coefficient',
    'has_transfer_history', 'max_career_transfer_fee', 'most_recent_transfer_fee',
    'detailed_position', 'foot', 'passport_tier',
]

PERFORMANCE_FEATURES = [
    'age', 'height_in_cm', 'total_appearances', 'minutes_per_match',
    'total_goals', 'total_assists', 'goals_per_90', 'assists_per_90',
    'total_yellow_cards', 'total_red_cards', 'international_caps', 'international_goals',
    'club_squad_size', 'club_avg_age', 'stadium_seats', 'foreigners_percentage',
    'wonderkid_hype', 'league_coefficient', 'detailed_position', 'foot', 'passport_tier',
]

NAME_MAP = {
    'remainder__age': 'Player Age',
    'remainder__max_career_transfer_fee': 'Max Transfer Fee',
    'remainder__most_recent_transfer_fee': 'Most Recent Transfer Fee',
    'remainder__international_caps': 'International Caps',
    'remainder__contract_months_remaining': 'Contract Remaining',
    'remainder__total_appearances': 'Total Appearances',
    'remainder__stadium_seats': 'Stadium Capacity',
    'remainder__minutes_per_match': 'Minutes Per Match',
    'remainder__foreigners_percentage': 'Foreigner Ratio',
    'remainder__has_transfer_history': 'Transfer History',
    'remainder__club_avg_age': 'Club Avg Age',
    'remainder__total_goals': 'Total Goals',
    'remainder__goals_per_90': 'Goals Per 90',
    'remainder__international_goals': 'International Goals',
    'remainder__club_squad_size': 'Squad Size',
    'remainder__total_assists': 'Total Assists',
    'remainder__height_in_cm': 'Height (cm)',
    'remainder__wonderkid_hype': 'Wonderkid Hype Index',
    'remainder__assists_per_90': 'Assists Per 90',
    'remainder__total_yellow_cards': 'Yellow Cards',
    'remainder__total_red_cards': 'Red Cards',
    'remainder__league_coefficient': 'League Quality',
    'cat__passport_tier_Tier_1': 'Passport (Tier 1)',
    'cat__detailed_position_Goalkeeper': 'Position: Goalkeeper',
}

def clean_feature_names(feature_names):
    return [
        NAME_MAP.get(name, name.replace('remainder__', '').replace('cat__', '').replace('_', ' ').title())
        for name in feature_names
    ]

# ==========================================
# 2. MAIN EXECUTION & INTERACTIVE LOOP
# ==========================================

def main():
    db_url = os.getenv('DB_URL')
    if not db_url:
        print("[ERROR] DB_URL not found. Please ensure your .env file exists and is configured correctly.")
        sys.exit(1)

    print("[SYSTEM] Connecting to database...")
    try:
        engine = create_engine(db_url)
        df = pd.read_sql("SELECT * FROM view_scout_master", engine)
    except Exception as e:
        print(f"[ERROR] Database connection failed: {e}")
        sys.exit(1)

    print("[SYSTEM] Performing robust feature engineering...")
    cols_to_clean = [
        'age', 'total_appearances', 'international_caps', 'international_goals',
        'max_career_transfer_fee', 'most_recent_transfer_fee', 'height_in_cm',
        'minutes_per_match', 'total_goals', 'total_assists', 'goals_per_90',
        'assists_per_90', 'total_yellow_cards', 'total_red_cards', 'club_squad_size',
        'club_avg_age', 'stadium_seats', 'foreigners_percentage', 'contract_months_remaining',
    ]
    
    for col in cols_to_clean:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors='coerce').fillna(0)

    df['wonderkid_hype'] = np.where(df['age'] <= 22, (df['total_appearances'] + (df['international_caps'] * 3)), 0)

    conditions = [
        (df['passport_power_rank'].fillna(999) <= 15),
        (df['passport_power_rank'].fillna(999) > 15) & (df['passport_power_rank'].fillna(999) <= 60),
    ]
    df['passport_tier'] = np.select(conditions, ['Tier_1', 'Tier_2'], default='Tier_3')

    league_weights = {
        'Premier League': 1.5, 'LaLiga': 1.4, 'Serie A': 1.3, 'Bundesliga': 1.3,
        'Ligue 1': 1.2, 'Eredivisie': 1.15, 'Liga Portugal': 1.15, 'Süper Lig': 1.05,
    }
    df['league_coefficient'] = df['competition_name'].map(league_weights).fillna(1.0)
    df['detailed_position'] = df['sub_position'].fillna(df['position_group']).astype(str)


    print("[SYSTEM] Loading models...")
    models = {}
    try:
        models["full"] = joblib.load(os.path.join(MODELS_DIR, 'scout_model_full.pkl'))
        models["performance_only"] = joblib.load(os.path.join(MODELS_DIR, 'scout_model_performance_only.pkl'))
    except FileNotFoundError as e:
        print(f"[ERROR] Could not load models from '{MODELS_DIR}'. Please train models first.")
        sys.exit(1)

    print("\n--- SCOUT AI INTERACTIVE TERMINAL ---")
    print("Available players (first 10):", df['player_name'].head(10).tolist())
    print("Type 'exit' or 'quit' to close the tool.")

    while True:
        try:
            target_player = input("\nEnter the player name to analyze (or 'exit'): ").strip()
            
            if not target_player:
                continue
            if target_player.lower() in ['exit', 'quit']:
                print("Exiting ScoutAI Inspector. Goodbye!")
                break

            player_data = df[df['player_name'].str.contains(target_player, case=False, na=False)]

            if player_data.empty:
                print(f"\n[ERROR] Player '{target_player}' not found. Please check the spelling.")
                continue
            
            # Warn if multiple matches, but proceed with the first one
            if len(player_data) > 1:
                print(f"\n[i] Found {len(player_data)} matches. Analyzing the first result: {player_data['player_name'].iloc[0]}")

            player = player_data.iloc[0]
            player_row = player_data.iloc[[0]]

            has_history = bool(player_row['has_transfer_history'].iloc[0])
            model_label = "full" if has_history else "performance_only"
            feature_list = FULL_FEATURES if has_history else PERFORMANCE_FEATURES

            print(f"[SYSTEM] Player has_transfer_history={int(has_history)} -> using '{model_label}' model.")

            pipeline = models[model_label]
            preprocessor = pipeline.named_steps['preprocessor']
            regressor = pipeline.named_steps['regressor']
            explainer = shap.TreeExplainer(regressor)
            clean_names = clean_feature_names(preprocessor.get_feature_names_out())

            X_player = player_row[feature_list]
            X_player_transformed = preprocessor.transform(X_player)
            shap_values_player = explainer(X_player_transformed)

            explanation = shap.Explanation(
                values=shap_values_player[0],
                base_values=explainer.expected_value,
                data=X_player_transformed[0],
                feature_names=clean_names,
            )

            plt.figure(figsize=(12, 8))
            shap.plots.waterfall(explanation, show=False)
            plt.title(
                f"ScoutAI Value Impact Analysis: {player['player_name']} ({model_label.replace('_', ' ').title()} model)",
                fontsize=16, fontweight='bold', pad=20,
            )
            plt.tight_layout()
            
            safe_name = player['player_name'].replace(' ', '_')
            out_path = os.path.join(IMAGES_DIR, f'shap_waterfall_{safe_name}.png')
            plt.savefig(out_path, dpi=150, bbox_inches='tight')
            plt.close() # Close the plot to free memory
            
            print(f"[SUCCESS] Waterfall plot successfully saved to '{out_path}'.")

        except KeyboardInterrupt:
            print("\nExiting ScoutAI Inspector. Goodbye!")
            break
        except Exception as e:
            print(f"\n[ERROR] An unexpected error occurred: {e}")

if __name__ == "__main__":
    main()

C:\Users\ali-k\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


[SYSTEM] Connecting to database...
[SYSTEM] Performing robust feature engineering...
[SYSTEM] Loading models...

--- SCOUT AI INTERACTIVE TERMINAL ---
Available players (first 10): ['James Milner', 'Óscar Ustari', 'Boy Waterman', 'Anastasios Tsokanis', 'Jonas Hofmann', 'Pepe Reina', 'Javier García', 'Cristiano Ronaldo', 'Philipp Pentke', 'David Marshall']
Type 'exit' or 'quit' to close the tool.
[SYSTEM] Player has_transfer_history=1 -> using 'full' model.
[SUCCESS] Waterfall plot successfully saved to 'images\shap_waterfall_Mohamed_Salah.png'.

[ERROR] Player 'EXİT' not found. Please check the spelling.
Exiting ScoutAI Inspector. Goodbye!
